### Synthetic Data Backtest

Most of the functions below can be found under:

* Tool/backtest 
* sample-data/make_data

A quick slideshow: [Seven Sins of Backtesting](https://newyork.qwafafew.org/wp-content/uploads/sites/4/2015/10/Luo_20150128.pdf)

The below is a code snippet for trading rules optimization.

The origin exercise required to use real-HFT data to deduce a certain outcome.. which I don't have.

Hence, I will just replace this exercise with multiprocessing methods (Parallelization!).

If you are keen on generating synthetic data for your research, copy the code snippets [Generate synthetic raw data](https://gist.github.com/boyboi86/5e00faf48f60abfdbe838fbdee269471) in my gist.

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cqrlib as rs
from random import gauss
from itertools import product

%matplotlib inline

In [ ]:
dict0 = {'a':['1', '2'], 'b':['+', '*'], 'c':['!','A']}
for a in dict0['a']:
    for b in dict0['b']:
        for c in dict0['c']:
            print({'a':a, 'b':b, 'c':c})

**Note**

The above is scalar operation. Notice loop after loop after loop.

The below is vectorization operation, which is what we will do for Monte-Carlos.

The result is actually the same.

In [ ]:
jobs = (dict(zip(dict0, i )) for i in product(*dict0.values()))
for i in jobs: print(i)

**Note**

Given the parameters below. It should look like the below.

In [ ]:
forecast = [10,5,0,-5,-10]
half_life = [5,10,25,50,100]
loss_range = profit_range = np.linspace(.5, 10, 20)
sigma = [1]
max_period = half_life[-1]
n_run = 100

coeff = dict({'forecast': forecast,
              'half_life': half_life,
              'sigma':sigma,
              'profit_range': profit_range,
              'loss_range':loss_range})
jobs = list(dict(zip(coeff, idx)) for idx in product(*coeff.values()))

print(len(jobs)) #forecast * half_life * loss * profit * sigma
out = pd.DataFrame(jobs)
out

**Note**

The below is the end product for a non-parallized Monte-Carlos Simulation, for optimizied Trading Rule Algorithm.

Primarily we will use it to create scenarios for our models to be backtest with.

There is another algorithm which MLfinlab has implemented, you may wish to read that as well.

[Backtesting by Campbell and Yan [2015]](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2345489)

In [ ]:
def _opt_tr(coeff: dict = None,
            n_run: float = 100,
            max_period: int = 100,
            seed: int = 0):
    
    phi, output1 = 2 ** (-1/ coeff['half_life']), []
    for iter_ in range(int(n_run)):
        p, hold_period, count = seed, 0, 0
        while True:
            p = (1 - phi) * coeff['forecast'] + phi * p + coeff['sigma'] * gauss(0,1)
            cp = p - seed
            hold_period += 1
            if coeff['profit_range'] < cp or -coeff['loss_range'] > cp or hold_period > max_period:
                output1.append(cp)
                break
    mean, std = np.mean(output1), np.std(output1)
    return {'profit': coeff['profit_range'], 'loss': coeff['loss_range'], 'mean': mean, 'std': std, 'sharpe': mean/std}

def opt_tr(profit_range: list = np.linspace(.5, 10, 20),
          loss_range: list = np.linspace(.5, 10, 20),
          sigma: list = [1],
          n_run: float = 100,
          max_period: int = 100,
          forecast: list = [10,5,0,-5,-10], 
          half_life: list = [5,10,25,50,100],
          num_threads: int = 1):
    
    if profit_range is None:
        profit_range = np.linspace(0,10,21)
        
    if loss_range is None:
        loss_range = np.linspace(0,10,21)
        
    _coeff_list = []
    _coeff = dict({'forecast': forecast,
                   'half_life': half_life,
                   'sigma':sigma,
                   'profit_range': profit_range,
                   'loss_range':loss_range})
    
    _coeff = list(dict(zip(_coeff, idx )) for idx in product(*_coeff.values()))
    out = []
    for coeff in _coeff:
        jobs = [{'func': _opt_tr,
                   'coeff': coeff,
                   'n_run': n_run, 
                   'max_period': max_period}]
        
        if num_threads == 1:
            out_ = rs.process_jobs_(jobs)
            out.append(out_)
        else:
            out_ = rs.process_jobs(jobs, num_threads = num_threads)
            out.append(out_)

    return out

df = opt_tr(profit_range = profit_range,
              loss_range = loss_range,
              sigma = sigma,
              n_run = n_run,
              max_period = max_period,
              forecast = forecast, 
              half_life = half_life,
              num_threads = 1)

**Note**

Before parallization. Total runs for estimated 1,000,000 loops took about 3 min

* 2020-06-17 14:41:28.739905 (Beginning!)

* 2020-06-17 14:46:51.676526 (End!)

**Note**

Try to change the above code snippet to run parallelization.

Please do not use jupyter notbook, it won't reflect anything.

If you are like me, curious about strategy's mean-reversion velocity, you might want to read this research report.

[Critical-Line Algorithm for Portfolio Optimization by Bailey and Lopez [2013]](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2197616)

**Hint**

Vectorize iterable then try using "iterrows()" or "itertuples()" if you are using pandas.

In [ ]:
print("Total count provided: {0}, total count retrieved: {1}".format(len(jobs), len(df)))
df

In [ ]:
df1 = rs.opt_tr(profit_range = profit_range,
                  loss_range = loss_range,
                  sigma = sigma,
                  n_run = n_run,
                  max_period = max_period,
                  forecast = forecast, 
                  half_life = half_life,
                  num_threads = 3)

In [ ]:
df1

**Conclusion**

The above was ran parallel with 3 threads:
    
1. 2020-06-17 14:49:16.186467
2. 2020-06-17 14:49:21.045204
3. 2020-06-17 14:49:46.331662

Based on the time provided by python system, it seems that it took less than 40 seconds.

When running large dataset, especially with Pandas parallelization can save time and always vectorize the datasets.

**Note**

If the answer to the above will not tally with the initial run (Scalar Operations), since we are running a Monte-Carlos.